# DMBI CA2 Mini Project: Employee Attrition Prediction Using Data Mining Techniques
**Student Name:** Prithviraj Virwani  
**Class:** D15B | **Roll No:** 70  
**Dataset Source (GitHub):** [WA_Fn-UseC_-HR-Employee-Attrition.csv](https://github.com/Prithvi696969/prithvi-dmbi-ca2-employee-attrition/blob/main/WA_Fn-UseC_-HR-Employee-Attrition.csv)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve

# Load Dataset directly from GitHub
URL = 'https://raw.githubusercontent.com/Prithvi696969/prithvi-dmbi-ca2-employee-attrition/main/WA_Fn-UseC_-HR-Employee-Attrition.csv'
df = pd.read_csv(URL)
print('Dataset Shape:', df.shape)
df.head()

## 2. Data Preprocessing & Encoding

In [ ]:
print('Missing Values:', df.isnull().sum().sum())
print('Duplicates:', df.duplicated().sum())

# Drop zero-variance and ID columns
df_clean = df.drop(columns=['EmployeeCount', 'Over18', 'StandardHours', 'EmployeeNumber'])

# Binary encoding
df_clean['Attrition'] = df_clean['Attrition'].map({'Yes': 1, 'No': 0})
df_clean['OverTime'] = df_clean['OverTime'].map({'Yes': 1, 'No': 0})
df_clean['Gender'] = df_clean['Gender'].map({'Male': 1, 'Female': 0})

# One-Hot Encoding
cat_cols = ['BusinessTravel', 'Department', 'EducationField', 'JobRole', 'MaritalStatus']
df_encoded = pd.get_dummies(df_clean, columns=cat_cols, drop_first=True)

X = df_encoded.drop('Attrition', axis=1)
y = df_encoded['Attrition']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 3. Data Mining Algorithms (Decision Tree, Random Forest, Logistic Regression)

In [ ]:
models = {
    'Decision Tree (CART)': (DecisionTreeClassifier(criterion='gini', max_depth=5, class_weight='balanced', random_state=42), False),
    'Random Forest (Ensemble)': (RandomForestClassifier(n_estimators=200, max_depth=8, class_weight='balanced', random_state=42), False),
    'Logistic Regression': (LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42), True)
}

summary = []
for name, (clf, scaled) in models.items():
    Xtr, Xte = (X_train_scaled, X_test_scaled) if scaled else (X_train, X_test)
    clf.fit(Xtr, y_train)
    pred = clf.predict(Xte)
    prob = clf.predict_proba(Xte)[:, 1]
    summary.append({
        'Model': name,
        'Accuracy (%)': round(accuracy_score(y_test, pred) * 100, 2),
        'Precision (%)': round(precision_score(y_test, pred) * 100, 2),
        'Recall (%)': round(recall_score(y_test, pred) * 100, 2),
        'F1-Score (%)': round(f1_score(y_test, pred) * 100, 2),
        'ROC-AUC': round(roc_auc_score(y_test, prob), 3)
    })

pd.DataFrame(summary)

## 4. Visualizations & Feature Importance

In [ ]:
rf = models['Random Forest (Ensemble)'][0]
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(8)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
imp.sort_values().plot.barh(color='#4f46e5', ax=axes[0])
axes[0].set_title('Top 8 Employee Attrition Predictors (Random Forest)')

cm = confusion_matrix(y_test, rf.predict(X_test))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['Pred: Stay', 'Pred: Leave'], yticklabels=['Actual: Stay', 'Actual: Leave'])
axes[1].set_title('Random Forest Confusion Matrix (N=294)')
plt.tight_layout()
plt.show()